# Financial Project Performance: ETL
**Flow:** Extract (Excel) → Validate (data quality checks) → Load (PostgreSQL tables) → Reconcile

Every problem found is tagged with one of three outcomes:
- **REJECTED**: row is not loaded (saved to `rejected_rows.csv`)
- **CORRECTED**: row is fixed by a rule and loaded
- **WARNING**: row is loaded unchanged but should be reviewed

## 1. Setup

In [11]:
import os
import re
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

EXCEL_FILE = "Financial_Performance_Case.xlsx"
DDL_FILE   = "01_ddl.sql"

VALID_CATEGORIES = ["Labor", "Materials", "Services", "Travel", "Contract"]
CATEGORY_FIXES = {          # known misspellings
    "Travell": "Travel", "Materiels": "Materials", "Labr": "Labor",
    "Contrct": "Contract", "Srvces": "Services",
}
VALID_MONTHS = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

issues = []     
rejected = []   

# Record a finding.
def log_issue(table, check, severity, rows, action):
    if len(rows) > 0:
        issues.append({"table": table, "check": check, "severity": severity,
                       "rows": len(rows), "action": action})

# keep a copy of Rejected rows 
def reject(rows, table, reason):
    if len(rows) > 0:
        rejected.append(rows.assign(source_table=table, reject_reason=reason))

## 2. Connect to PostgreSQL Database
Credentials are read from a **`.env` file** in the same folder as this notebook, so no password is stored in the notebook.


In [12]:
env_path = find_dotenv(usecwd=True)          
if not env_path:
    raise FileNotFoundError(".env file not found.")
load_dotenv(env_path)                         


db_url = URL.create(
    drivername="postgresql+psycopg2",
    host=os.environ["PGHOST"],
    port=int(os.environ.get("PGPORT", 5432)),
    database=os.environ["PGDATABASE"],
    username=os.environ["PGUSER"],
    password=os.environ["PGPASSWORD"],      
)
SCHEMA = os.environ.get("PGSCHEMA", "public")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", SCHEMA):
    raise ValueError(f"Invalid schema name: {SCHEMA!r} (letters, digits and underscores only)")

engine = create_engine(
    db_url,
    connect_args={
        "sslmode": os.environ.get("PGSSLMODE", "require"),
        "connect_timeout": 10,
        "options": f"-c search_path={SCHEMA}",   
    },
)


with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar())
    print("Connected to database:", conn.execute(text("SELECT current_database()")).scalar())
    print("Working schema:", SCHEMA)

PostgreSQL 18.3 on aarch64-unknown-linux-gnu, compiled by aarch64-unknown-linux-gnu-gcc (GCC) 12.4.0, 64-bit
Connected to database: postgres
Working schema: finance


## 3. Extract

In [13]:
sheets = pd.read_excel(EXCEL_FILE, sheet_name=None)
projects, transactions, forecasts = sheets["Projects"], sheets["Transactions"], sheets["Forecasts"]
source_counts = {"projects": len(projects), "transactions": len(transactions), "forecasts": len(forecasts)}
source_counts

{'projects': 50, 'transactions': 516, 'forecasts': 150}

## 4. Validate and clean
### 4.1 Projects

In [14]:
# Trim spaces on text columns
for col in ["Project_ID", "Project_Name", "Region", "Manager"]:
    projects[col] = projects[col].str.strip()

# Duplicate Project_ID
dup = projects[projects.duplicated("Project_ID", keep="first")]
log_issue("Projects", "Duplicate Project_ID", "REJECTED", dup, "Keep first, reject the rest")
reject(dup, "Projects", "Duplicate Project_ID")
projects = projects.drop(dup.index)

# Budget must be a positive number
projects["Budget"] = pd.to_numeric(projects["Budget"], errors="coerce")
bad = projects[~(projects["Budget"] > 0)]
log_issue("Projects", "Budget missing or <= 0", "REJECTED", bad, "Row rejected")
reject(bad, "Projects", "Budget missing or <= 0")
projects = projects.drop(bad.index)

len(projects)

50

### 4.2 Transactions

In [ ]:
transactions["Amount"] = pd.to_numeric(transactions["Amount"], errors="coerce")
transactions["Transaction_Date"] = pd.to_datetime(transactions["Transaction_Date"], errors="coerce")

# 1) Exact duplicate rows (keep the first copy)
dup = transactions[transactions.duplicated(keep="first")]
log_issue("Transactions", "Exact duplicate rows", "REJECTED", dup, "Keep first copy, reject the rest")
reject(dup, "Transactions", "Exact duplicate row")
transactions = transactions.drop(dup.index)

# 2) Missing Project_ID
missing = transactions[transactions["Project_ID"].isna()]
log_issue("Transactions", "Missing Project_ID", "REJECTED", missing, "Row rejected")
reject(missing, "Transactions", "Missing Project_ID")
transactions = transactions.drop(missing.index)

# 3) Project_ID that does not exist in Projects
orphan = transactions[~transactions["Project_ID"].isin(projects["Project_ID"])]
log_issue("Transactions", "Project_ID not in Projects", "REJECTED", orphan, "Row rejected")
reject(orphan, "Transactions", "Project_ID not found in Projects")
transactions = transactions.drop(orphan.index)

# 4) Misspelled categories 
fixed = transactions[transactions["Category"].isin(CATEGORY_FIXES)]
log_issue("Transactions", "Misspelled Category", "CORRECTED", fixed,
          "Mapped to standard value ('Contrct' assumed to be 'Contract')")
transactions["Category"] = transactions["Category"].replace(CATEGORY_FIXES)

# 5) Anything still not a valid category, or missing amount/date
unknown = transactions[~transactions["Category"].isin(VALID_CATEGORIES)
                       | transactions["Amount"].isna() | (transactions["Amount"] == 0)
                       | transactions["Transaction_Date"].isna()]
log_issue("Transactions", "Unknown category / bad amount / bad date", "REJECTED", unknown, "Row rejected")
reject(unknown, "Transactions", "Unknown category, bad amount or bad date")
transactions = transactions.drop(unknown.index)

# 6) Negative amounts: keep (could be a credit) but flag
negative = transactions[transactions["Amount"] < 0]
log_issue("Transactions", "Negative Amount", "WARNING", negative, "Loaded as credit, confirm with finance")

len(transactions)

501

### 4.3 Forecasts

In [16]:
# Invalid month
bad = forecasts[~forecasts["Month"].isin(VALID_MONTHS)]
log_issue("Forecasts", "Invalid Month", "REJECTED", bad, "Row rejected")
reject(bad, "Forecasts", "Invalid Month")
forecasts = forecasts.drop(bad.index)

# Duplicate (Project_ID, Month)
dup = forecasts[forecasts.duplicated(["Project_ID", "Month"], keep="first")]
log_issue("Forecasts", "Duplicate Project_ID + Month", "REJECTED", dup, "Keep first, reject the rest")
reject(dup, "Forecasts", "Duplicate Project_ID + Month")
forecasts = forecasts.drop(dup.index)

# Missing/negative amount, or project not in Projects
bad = forecasts[~(forecasts["Forecast_Amount"] >= 0) | ~forecasts["Project_ID"].isin(projects["Project_ID"])]
log_issue("Forecasts", "Bad amount or unknown Project_ID", "REJECTED", bad, "Row rejected")
reject(bad, "Forecasts", "Bad amount or unknown Project_ID")
forecasts = forecasts.drop(bad.index)

len(forecasts)

150

## 5. Data quality report

In [17]:
report = pd.DataFrame(issues)
report.to_csv("data_quality_report.csv", index=False)
report

,table,check,severity,rows,action
0,Transactions,Exact duplicate rows,REJECTED,9,"Keep first copy, reject the rest"
1,Transactions,Missing Project_ID,REJECTED,1,Row rejected
2,Transactions,Project_ID not in Projects,REJECTED,5,Row rejected
3,Transactions,Misspelled Category,CORRECTED,5,Mapped to standard value ('Contrct' assumed to...
4,Transactions,Negative Amount,WARNING,1,"Loaded as credit, confirm with finance"


In [18]:
rejected_rows = pd.concat(rejected, ignore_index=True)
rejected_rows.to_csv("rejected_rows.csv", index=False)
rejected_rows.groupby(["source_table", "reject_reason"]).size().rename("rows").reset_index()

,source_table,reject_reason,rows
0,Transactions,Exact duplicate row,9
1,Transactions,Missing Project_ID,1
2,Transactions,Project_ID not found in Projects,5


## 6. Load


In [19]:
transactions["Transaction_Date"] = transactions["Transaction_Date"].dt.date   # DATE column

with open(DDL_FILE) as f:
    ddl_lines = [line for line in f if not line.strip().startswith("--")]
ddl_statements = [s.strip() for s in "".join(ddl_lines).split(";") if s.strip()]

with engine.begin() as conn:
    conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{SCHEMA}"'))
    for stmt in ddl_statements:
        conn.execute(text(stmt))

    for name, df in [("projects", projects), ("transactions", transactions), ("forecasts", forecasts)]:
        df = df.copy()
        df.columns = df.columns.str.lower()
        df.to_sql(name, conn, schema=SCHEMA, if_exists="append", index=False, method="multi", chunksize=500)
        print(f"loaded {len(df):>4} rows into {SCHEMA}.{name}")

loaded   50 rows into finance.projects
loaded  501 rows into finance.transactions
loaded  150 rows into finance.forecasts


## 7. Reconcile
Source rows must equal loaded rows plus rejected rows.

In [20]:
rejected_counts = rejected_rows["source_table"].str.lower().value_counts().to_dict()

with engine.connect() as conn:
    loaded = {t: conn.execute(text(f'SELECT COUNT(*) FROM "{SCHEMA}".{t}')).scalar()
              for t in ["projects", "transactions", "forecasts"]}

check = pd.DataFrame([{
    "table": t,
    "source_rows": source_counts[t],
    "rejected": rejected_counts.get(t, 0),
    "loaded_in_db": loaded[t],
} for t in loaded])
check["ok"] = check["source_rows"] == check["rejected"] + check["loaded_in_db"]
engine.dispose()
check

,table,source_rows,rejected,loaded_in_db,ok
0,projects,50,0,50,True
1,transactions,516,15,501,True
2,forecasts,150,0,150,True
